# 🍨 Fine-Tuning da Naiara (Recanto do Açaí) com Unsloth 🚀

Este notebook demonstra o processo passo a passo para fazer o fine-tuning (LoRA) do modelo **Qwen 2.5 7B Instruct** utilizando a biblioteca super otimizada **Unsloth**.

### Objetivo
Treinar a **Naiara** (assistente comercial virtual do Recanto do Açaí) para:
1. Adotar o tom de voz comercial carioca, simpático e vendedor.
2. Responder no formato correto do WhatsApp, utilizando o delimitador `[MSG_BREAK]` para separar mensagens consecutivas.
3. Chamar ferramentas JSON (Tool Calling) para checar disponibilidade de datas, efetuar reservas temporárias no CRM, atualizar lead scores, aplicar etiquetas de funil ou acionar o transbordo humano (`chamar_humano`).

---

## 1. Instalar Dependências no Google Colab / Unsloth Studio 📦

In [ ]:
# Instalação do Unsloth e dependências otimizadas para treinamento acelerado
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "peft" "accelerate" "bitsandbytes"
!pip install datasets trl transformers huggingface_hub

## 2. Autenticação no Hugging Face Hub 🔑

In [ ]:
# Definir variáveis do Hugging Face
HF_TOKEN = os.environ.get("HF_TOKEN", "")

import os
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN)

## 3. Carregar o Modelo e Tokenizador via Unsloth 🧠

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # Tamanho máximo do contexto
dtype = None         # Detecção automática (float16 / bfloat16)
load_in_4bit = True  # Habilitar quantização de 4 bits (economia dramática de VRAM)

print("⏳ Carregando o modelo base otimizado...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 4. Configurar Parâmetros de Adaptação (LoRA) ⚙️

In [ ]:
print("⚙️ Configurando adaptadores LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank do LoRA (16 ou 32)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Totalmente otimizado para 0
    bias = "none",    # Otimizado para "none"
    use_gradient_checkpointing = "unsloth", # Economiza ainda mais VRAM
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 5. Carregar e Formatar o Dataset de Diálogos 📊

In [ ]:
from datasets import load_dataset

DATASET_REPO = "Finish-him/naiara-recanto-dataset"

# Função para formatar os diálogos no template oficial de chat do Qwen 2.5 (ChatML)
def format_prompts(examples):
    formatted_texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted_texts.append(text)
    return { "text" : formatted_texts }

print("📊 Carregando o dataset do HF Hub...")
dataset = load_dataset(DATASET_REPO, split="train")
dataset = dataset.map(format_prompts, batched=True)

print(f"✅ {len(dataset)} amostras de diálogos carregadas e formatadas com sucesso!")

## 6. Configurar o Trainer (SFTTrainer) e Treinar! 🏋️

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Número de iterações de fine-tuning
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Iniciando treinamento...")
trainer_stats = trainer.train()
print("🎉 Treinamento finalizado!")

## 7. Salvar e Enviar os Adaptadores Treinados para o Hub 💾

In [ ]:
HF_MODEL_REPO = "Finish-him/qwen2.5-7b-naiara-lora"

print("💾 Salvando adaptadores locais...")
model.save_pretrained("lora_adapters")
tokenizer.save_pretrained("lora_adapters")

print(f"🚀 Enviando adaptadores LoRA para Hugging Face Hub em '{HF_MODEL_REPO}'...")
model.push_to_hub(HF_MODEL_REPO, private=True)
tokenizer.push_to_hub(HF_MODEL_REPO, private=True)

print(f"🎉 Concluído com sucesso! Link: https://huggingface.co/{HF_MODEL_REPO}")